# 🧠 RAG Chatbot: Chat With Your Own Notes/PDFs

A weekend-project-friendly RAG pipeline. No local install, no GPU needed, runs entirely in this Colab notebook.

**Stack:**
- LLM: Google Gemini API (free tier, rate-limited not trial-limited — won't expire)
- Embeddings: `sentence-transformers` (local, free, small model)
- Vector store: ChromaDB (in-memory, lightweight)
- PDF parsing: PyPDF2

**Before you start:**
1. Get a free Gemini API key here: https://aistudio.google.com/apikey (takes ~1 min, just needs a Google account, no credit card)
2. Have 1-3 small PDFs or .txt files ready (your notes, a manual, whatever you want to chat with)

Run the cells top to bottom. Each cell is a step — read the comments, they explain the 'why'.

## Step 1: Install dependencies
This installs everything into the Colab VM only — nothing touches your own machine's storage.

In [1]:
!pip install -q google-generativeai chromadb sentence-transformers pypdf2 tiktoken

In [2]:
import google.generativeai as genai
import chromadb
from sentence_transformers import SentenceTransformer
import PyPDF2
import tiktoken

print("Everything installed!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Everything installed!


## Step 2: Set your Gemini API key
Paste your key below. It's only stored in this notebook's runtime memory — nothing is saved to disk.

In [3]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key: ")

Paste your Gemini API key: ··········


## Step 3: Upload your documents
Upload 1-3 PDFs or .txt files. Since your dataset is small, this whole pipeline will be fast and light on memory.

In [4]:
from google.colab import files

uploaded = files.upload()
uploaded_filenames = list(uploaded.keys())
print("Uploaded:", uploaded_filenames)

Saving Unit 1.pdf to Unit 1 (1).pdf
Uploaded: ['Unit 1 (1).pdf']


## Step 4: Extract text from your documents
Handles both PDF and plain text files.

In [5]:
from PyPDF2 import PdfReader

def extract_text(filename):
    if filename.lower().endswith(".pdf"):
        reader = PdfReader(filename)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        return text
    else:
        with open(filename, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

raw_documents = {}
for fname in uploaded_filenames:
    raw_documents[fname] = extract_text(fname)
    print(f"{fname}: {len(raw_documents[fname])} characters extracted")

Unit 1 (1).pdf: 57572 characters extracted


## Step 5: Chunk the text
LLMs and embeddings work best on small chunks, not whole documents. We split into overlapping chunks so context isn't lost at chunk boundaries.

`chunk_size=500` and `overlap=50` are good defaults for small note-style documents. Feel free to tweak later.

In [6]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return [c.strip() for c in chunks if c.strip()]

all_chunks = []
all_metadata = []

for fname, text in raw_documents.items():
    doc_chunks = chunk_text(text)
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append(chunk)
        all_metadata.append({"source": fname, "chunk_index": i})

print(f"Total chunks created: {len(all_chunks)}")
print("\nExample chunk:\n", all_chunks[0][:300] if all_chunks else "No chunks found")

Total chunks created: 128

Example chunk:
 CS1701 A        Machine Learning                                              Dr.R.Geetha/Professor -                                                          Department of CSE                                                   Unit I                            1 
  
 
 
 
Definition of machine learn


## Step 6: Generate embeddings and store in ChromaDB
`all-MiniLM-L6-v2` is a small (~80MB), fast, free embedding model — good enough for a personal notes chatbot. Everything is stored in-memory for this session, so nothing eats into your disk space.

In [7]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.Client()  # in-memory, resets when runtime resets
collection = chroma_client.get_or_create_collection(name="my_notes")

embeddings = embedder.encode(all_chunks).tolist()

collection.add(
    documents=all_chunks,
    embeddings=embeddings,
    metadatas=all_metadata,
    ids=[f"chunk_{i}" for i in range(len(all_chunks))]
)

print(f"Stored {len(all_chunks)} chunks in the vector database.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Stored 128 chunks in the vector database.


## Step 7: Retrieval function
Given a question, find the most relevant chunks from your documents.

In [8]:
def retrieve_relevant_chunks(query, top_k=3):
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    return results["documents"][0], results["metadatas"][0]

# Quick test
test_chunks, test_meta = retrieve_relevant_chunks("What is supervised learning?")
for c, m in zip(test_chunks, test_meta):
    print(f"[{m['source']}] {c[:150]}...\n")

[Unit 1 (1).pdf] Dr.R.Geetha/Professor -                                                          Department of CSE                                                   U...

[Unit 1 (1).pdf] eled data to train machine 
learning models.  
Working:  
• Supervised learning algorithms takes labeled inputs and map them to the known outputs, whi...

[Unit 1 (1).pdf] d data.  
• We can imagine these algorithms with an example.  
• Supervised learning is where a student is under the supervision of an instructor at h...



## Step 8: The RAG chat function
This is the core of the whole project: retrieve relevant chunks, stuff them into the prompt as context, and ask the LLM to answer using only that context. This is what makes it 'RAG' rather than a plain chatbot.

In [9]:
import google.generativeai as genai

genai.configure(api_key=os.environ["GEMINI_API_KEY"])

def ask_rag_chatbot(question, top_k=3, model="gemini-2.5-flash-lite"):
    chunks, metadata = retrieve_relevant_chunks(question, top_k=top_k)

    context = "\n\n---\n\n".join(
        f"[Source: {m['source']}]\n{c}" for c, m in zip(chunks, metadata)
    )

    system_prompt = (
        "You are a helpful assistant that answers questions using ONLY the provided context. "
        "If the answer isn't in the context, say you don't know based on the provided documents. "
        "Cite which source file you're drawing from when relevant."
    )

    full_prompt = f"{system_prompt}\n\nContext:\n{context}\n\nQuestion: {question}"

    gemini_model = genai.GenerativeModel(model)
    response = gemini_model.generate_content(
        full_prompt,
        generation_config=genai.types.GenerationConfig(temperature=0.3)
    )

    # Defensive check: Gemini can return zero candidates (e.g. safety block)
    # without raising an exception -- response.text would then be empty or error.
    if not response.candidates:
        print("DEBUG: no candidates returned. prompt_feedback:", response.prompt_feedback)
        return "I couldn't generate a response for that question (it may have been filtered). Try rephrasing it.", metadata

    finish_reason = response.candidates[0].finish_reason
    if finish_reason != 1:  # 1 = STOP (normal completion)
        print(f"DEBUG: finish_reason={finish_reason}, full response: {response}")

    answer_text = response.text.strip() if response.text else ""
    if not answer_text:
        answer_text = "I got an empty response from the model. Check the cell output above for a DEBUG line, or try rephrasing your question."

    return answer_text, metadata

# Try it out
answer, sources = ask_rag_chatbot("Summarize the key points of these documents.")
print("ANSWER:\n", answer)
print("\nSOURCES USED:", [m["source"] for m in sources])

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1661.31ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1559.89ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1763.38ms


ANSWER:
 The provided documents discuss Machine Learning, specifically focusing on its components and a credit scoring example.

Key points include:

*   **Machine Learning Goal:** To convert knowledge about stored data into a usable form for future actions on similar, but not identical, tasks. This process is referred to as generalization. [Source: Unit 1 (1).pdf]
*   **Components of the Learning Process:** The learning process can be divided into four components: data storage, abstraction, generalization, and evaluation. [Source: Unit 1 (1).pdf]
*   **Data Storage:** Facilities for storing and retrieving large amounts of data are crucial for the learning process, serving as a foundation for advanced reasoning for both humans and computers. [Source: Unit 1 (1).pdf]
*   **Credit Scoring Example:** This involves differentiating between low-risk and high-risk customers based on information like income and savings. Banks use customer data such as income, savings, collaterals, profession, 

## Step 9: Interactive chat loop
Run this cell and type your questions. Type `exit` to stop.

In [10]:
print("RAG Chatbot ready. Type 'exit' to stop.\n")

while True:
    question = input("You: ")
    if question.lower().strip() == "exit":
        print("Goodbye!")
        break
    answer, sources = ask_rag_chatbot(question)
    print(f"\nBot: {answer}")
    print(f"(sources: {[m['source'] for m in sources]})\n")

RAG Chatbot ready. Type 'exit' to stop.

You: what is deep learning?

Bot: I don't know based on the provided documents.
(sources: ['Unit 1 (1).pdf', 'Unit 1 (1).pdf', 'Unit 1 (1).pdf'])

You: what is machine learning?


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3118.72ms



Bot: Machine learning is defined as "the field of study that gives computers the ability to learn without being explicitly programmed." [Source: Unit 1 (1).pdf] It is also described as programming computers to optimize a performance criterion using example data or past experience, where learning involves executing a computer program to optimize model parameters using training data or past experience. [Source: Unit 1 (1).pdf] Another definition states that machine learning is concerned with how to construct computer programs that automatically improve with experience. [Source: Unit 1 (1).pdf] However, there is no universally accepted definition for machine learning, and different authors define it differently. [Source: Unit 1 (1).pdf]
(sources: ['Unit 1 (1).pdf', 'Unit 1 (1).pdf', 'Unit 1 (1).pdf'])

You: exit
Goodbye!


## Step 10 (optional): A real chat UI with Gradio
The `input()` loop above works but doesn't look like a chatbot. Gradio gives you an actual chat interface — message bubbles, a text box, chat history — rendered right in Colab, plus a public shareable link (`share=True`) you can send to anyone, no deployment needed.

This reuses the `ask_rag_chatbot` function and `collection` you already built above — no need to re-upload or reprocess anything.

In [ ]:
import gradio as gr

def gradio_chat_fn(message, history):
    try:
        answer, metadata = ask_rag_chatbot(message)
        print(f"DEBUG: question={message!r} | answer_len={len(answer) if answer else 0}")
        if not answer or not answer.strip():
            return "⚠️ Got an empty answer back. Check the cell output above for DEBUG lines."
        sources = sorted(set(m["source"] for m in metadata))
        return f"{answer}\n\n*Sources: {', '.join(sources)}*"
    except Exception as e:
        import traceback
        traceback.print_exc()
        return f"⚠️ Something went wrong: `{e}`\n\nCheck that you've re-run all the setup cells above (API key, document upload, embeddings) in this Colab session."

gr.close_all()

demo = gr.ChatInterface(
    fn=gradio_chat_fn,
    title="📚 Chat with your notes",
    description="Ask questions grounded in the documents you uploaded above.",
    examples=["Summarize the key points", "What does this document say about..."],
)

# share=True gives you a public URL that works for ~72 hours, great for demos.
# debug=True prints any errors from gradio_chat_fn directly in this cell's output.
demo.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://358cb9d437ceaf33df.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


DEBUG: question='types of machine learning' | answer_len=344
